In [1]:
import torch
print('torch     ', torch.__version__)
print('cuda build', torch.version.cuda)
print('gpu       ', torch.cuda.get_device_name(0))
print('capability', torch.cuda.get_device_capability(0))
device = torch.device('cpu')

torch      2.10.0+cu128
cuda build 12.8
gpu        Tesla P100-PCIE-16GB
capability (6, 0)


/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 Tesla P100-PCIE-16GB which is of cuda capability 6.0.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.0) - (12.0)
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.6 following instructions at
    https://pytorch.org/get-started/locally/
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
Tesla P100-PCIE-16GB with CUDA capability sm_60 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_70 sm_75 sm_80 sm_86 sm_90 sm_100 sm_120.
If you want to use the Tesla P100-PCIE-16GB GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  queued_call()


In [2]:
!pip install -q --no-deps "timm>=1.0.20"

In [3]:
# NOTEBOOK 2 -- HEAD EXPERIMENTS (standalone)
# Cell 1 below IS csiro_metric.py, pasted in. Nothing to upload separately.
# Attach only: the competition data + your nb1 notebook output.

import numpy as np
import pandas as pd
from sklearn.metrics import r2_score
from sklearn.model_selection import StratifiedGroupKFold

TARGETS = ["Dry_Green_g", "Dry_Dead_g", "Dry_Clover_g", "GDM_g", "Dry_Total_g"]

WEIGHTS = {
    "Dry_Green_g": 0.1,
    "Dry_Dead_g": 0.1,
    "Dry_Clover_g": 0.1,
    "GDM_g": 0.2,
    "Dry_Total_g": 0.5,
}

META_COLS = ["State", "Species", "Sampling_Date", "Pre_GSHH_NDVI", "Height_Ave_cm"]


# ----------------------------------------------------------------------
# reshaping
# ----------------------------------------------------------------------

def to_wide(long_df):
    """
    Long format (one row per image-target pair) -> wide (one row per image).

    Returns a frame indexed by base_id with the 5 target columns plus whatever
    metadata columns are present. image_path is carried through.
    """
    df = long_df.copy()
    if "base_id" not in df.columns:
        df["base_id"] = df["sample_id"].str.split("__").str[0]

    wide = df.pivot(index="base_id", columns="target_name", values="target")
    wide = wide[[c for c in TARGETS if c in wide.columns]]
    wide.columns.name = None

    keep = ["image_path"] + [c for c in META_COLS if c in df.columns]
    meta = df.groupby("base_id")[keep].first()

    out = meta.join(wide)

    # metadata must be constant within an image, or the pivot hid a problem
    for c in [c for c in META_COLS if c in df.columns]:
        n = df.groupby("base_id")[c].nunique()
        if (n > 1).any():
            raise ValueError(f"{c} varies within a base_id -- check the source data")

    return out


# ----------------------------------------------------------------------
# the metric
# ----------------------------------------------------------------------

def weighted_log_r2(y_true, y_pred, verbose=False):
    """
    THE competition metric.

    y_true, y_pred: DataFrames (or arrays) of shape (n_images, 5) in GRAMS,
    columns ordered as TARGETS. log1p is applied internally -- pass grams.

    Returns (score, per_target_dict).
    """
    yt = _as_frame(y_true)
    yp = _as_frame(y_pred)

    if yt.shape != yp.shape:
        raise ValueError(f"shape mismatch: true {yt.shape} vs pred {yp.shape}")
    if (yp.values < 0).any():
        raise ValueError("negative predictions -- clip at 0 before scoring")

    per_target = {}
    score = 0.0
    for t in TARGETS:
        r2 = r2_score(np.log1p(yt[t].values), np.log1p(yp[t].values))
        per_target[t] = r2
        score += WEIGHTS[t] * r2

    if verbose:
        print(f"{'target':14} {'weight':>7} {'R2(log)':>9} {'contrib':>9}")
        for t in TARGETS:
            print(f"{t:14} {WEIGHTS[t]:>7.2f} {per_target[t]:>9.4f} "
                  f"{WEIGHTS[t] * per_target[t]:>9.4f}")
        print(f"{'':14} {'':>7} {'':>9} {'-' * 9}")
        print(f"{'WEIGHTED':14} {'':>7} {'':>9} {score:>9.4f}")

    return score, per_target


def pooled_r2_grams(y_true, y_pred):
    """
    The OLD metric -- one global R2 over all rows in grams, all 5 targets mixed.
    Kept only for comparison. Reads high because it gets credit for knowing
    that Clover averages ~6.7g and Total averages ~45.3g.
    """
    yt = _as_frame(y_true)[TARGETS].values.ravel()
    yp = _as_frame(y_pred)[TARGETS].values.ravel()
    return r2_score(yt, yp)


def _as_frame(x):
    if isinstance(x, pd.DataFrame):
        missing = [t for t in TARGETS if t not in x.columns]
        if missing:
            raise ValueError(f"missing target columns: {missing}")
        return x[TARGETS]
    arr = np.asarray(x)
    if arr.ndim != 2 or arr.shape[1] != 5:
        raise ValueError(f"expected (n, 5), got {arr.shape}")
    return pd.DataFrame(arr, columns=TARGETS)


def score_long_predictions(long_df, pred_col="pred", true_col="target"):
    """
    Score predictions that are still in long format -- so you can run your
    existing ConvNeXt checkpoint through the real metric without restructuring
    anything first.

    long_df needs: sample_id (or base_id) + target_name + true_col + pred_col.
    """
    df = long_df.copy()
    if "base_id" not in df.columns:
        df["base_id"] = df["sample_id"].str.split("__").str[0]

    yt = df.pivot(index="base_id", columns="target_name", values=true_col)
    yp = df.pivot(index="base_id", columns="target_name", values=pred_col)

    if yt.isna().any().any() or yp.isna().any().any():
        raise ValueError("pivot produced NaN -- some image is missing a target row")

    yp = yp.clip(lower=0)
    return weighted_log_r2(yt, yp, verbose=True)


# ----------------------------------------------------------------------
# folds
# ----------------------------------------------------------------------

def make_folds(wide_df, n_splits=5, seed=42, verbose=True):
    """
    StratifiedGroupKFold: stratify on State, group on Sampling_Date.

    Grouping on date is the point -- images from one sampling session share
    site, weather and species, so splitting them across folds leaks.

    Returns an int array of fold ids aligned to wide_df's rows.
    """
    for c in ("State", "Sampling_Date"):
        if c not in wide_df.columns:
            raise ValueError(f"need column {c} to build grouped folds")

    y = wide_df["State"].values
    groups = pd.to_datetime(wide_df["Sampling_Date"]).dt.strftime("%Y-%m-%d").values

    n_groups = len(np.unique(groups))
    if n_groups < n_splits:
        raise ValueError(f"{n_groups} sampling dates < {n_splits} folds")

    folds = np.full(len(wide_df), -1, dtype=int)
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for k, (_, val_idx) in enumerate(sgkf.split(wide_df, y, groups)):
        folds[val_idx] = k

    if (folds < 0).any():
        raise RuntimeError("some rows were never assigned to a fold")

    _check_fold_integrity(folds, groups)

    if verbose:
        print(f"{n_groups} sampling dates -> {n_splits} folds "
              f"({len(wide_df)} images)\n")
        comp = pd.crosstab(folds, wide_df["State"].values)
        comp.index.name = "fold"
        print(comp)
        print(f"\nimages per fold: {np.bincount(folds).tolist()}")

    return folds


def _check_fold_integrity(folds, groups):
    """A sampling date must live in exactly one fold. Hard assert."""
    spans = pd.DataFrame({"fold": folds, "group": groups}) \
              .groupby("group")["fold"].nunique()
    bad = spans[spans > 1]
    if len(bad):
        raise RuntimeError(
            f"{len(bad)} sampling dates span multiple folds -- leak: "
            f"{list(bad.index[:5])}"
        )


# ----------------------------------------------------------------------
# the reference floor
# ----------------------------------------------------------------------

def mean_baseline(wide_df, folds, verbose=True):
    """
    Predict each target's TRAIN-fold mean. This is where zero is.

    Any model must clear this by a visible margin. Had this existed earlier it
    would have caught the gray-image run: a backbone fed constant input scores
    the same as this, because that is literally what it computes.

    Predicting the train mean (not the val mean) lands at or slightly below 0.
    """
    oof = pd.DataFrame(index=wide_df.index, columns=TARGETS, dtype=float)

    for k in range(folds.max() + 1):
        tr, va = folds != k, folds == k
        # mean in log space, since the metric is computed there
        means = np.expm1(np.log1p(wide_df.loc[tr, TARGETS]).mean())
        oof.loc[va, TARGETS] = means.values

    score, per_target = weighted_log_r2(wide_df[TARGETS], oof, verbose=verbose)
    if verbose:
        print(f"\npooled R2 in grams, same predictions: "
              f"{pooled_r2_grams(wide_df[TARGETS], oof):.4f}   <-- the old metric")
    return score, per_target, oof


# ----------------------------------------------------------------------

def load_and_prepare(data_dir="/kaggle/input/csiro-biomass", n_splits=5, seed=42):
    """train.csv -> (wide_df, folds). One call to set up an experiment."""
    long_df = pd.read_csv(f"{data_dir}/train.csv")
    wide = to_wide(long_df)
    print(f"{len(long_df)} rows -> {len(wide)} images\n")
    folds = make_folds(wide, n_splits=n_splits, seed=seed)
    return wide, folds

print('metric functions loaded')

metric functions loaded


In [4]:
# NOTEBOOK 2 of 4 -- HEAD EXPERIMENTS ON CACHED FEATURES
#
# Every design question answered here costs under a second of GPU.
# This is the notebook that replaces the runs that were burning hours.
#
# SETTINGS: Accelerator = GPU P100 | Internet = OFF | attach nb1 output
#           and csiro_metric.py (as a utility script or dataset)

import json, time, itertools
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

DATA_DIR = '/kaggle/input/competitions/csiro-biomass'
FEAT_DIR = '/kaggle/input/datasets/upashanad/csironb1features'     # <-- set this

device = torch.device('cuda')
print('device:', torch.cuda.get_device_name(0))

device: Tesla P100-PCIE-16GB


In [5]:
# Load labels and the feature cache, and assert they are aligned.
# Misalignment here would be silent and fatal, so it is checked, not assumed.

long_df = pd.read_csv(f'{DATA_DIR}/train.csv')
wide    = to_wide(long_df)

feat_ids = pd.read_csv(f'{FEAT_DIR}/feature_ids.csv')['base_id'].tolist()
F        = np.load(f'{FEAT_DIR}/features.npy')          # (n_img, 4, 2*DIM)
manifest = json.load(open(f'{FEAT_DIR}/manifest.json'))

wide = wide.loc[feat_ids]                                # align to feature order
assert list(wide.index) == feat_ids, 'label/feature misalignment'
assert F.shape[0] == len(wide), (F.shape, len(wide))

print(json.dumps(manifest, indent=2))
print(f'\nfeatures {F.shape}  labels {wide[TARGETS].shape}')

folds = make_folds(wide, n_splits=5, seed=42)

{
  "model": "vit_huge_plus_patch16_dinov3.lvd1689m",
  "img_size": 800,
  "feature_dim": 1280,
  "pooled": [
    "cls",
    "patch_mean"
  ],
  "vector_dim": 2560,
  "views": [
    "id",
    "h",
    "v",
    "hv"
  ],
  "n_images": 357,
  "mean": [
    0.485,
    0.456,
    0.406
  ],
  "std": [
    0.229,
    0.224,
    0.225
  ],
  "shape": [
    357,
    4,
    2560
  ],
  "seconds": 1257.5
}

features (357, 4, 2560)  labels (357, 5)
28 sampling dates -> 5 folds (357 images)

col_0  NSW  Tas  Vic  WA
fold                    
0       10   27   19  12
1       28    9    0   8
2        0   25   27   0
3       24   42   16  12
4       13   35   50   0

images per fold: [68, 45, 52, 94, 98]


In [6]:
# THE FLOOR. Run this before anything else and write the number down.
# Any head that does not clearly beat it has learned nothing.

print('=' * 64)
print('MEAN BASELINE -- where zero is')
print('=' * 64)
base_score, _, _ = mean_baseline(wide, folds)

MEAN BASELINE -- where zero is
target          weight   R2(log)   contrib
Dry_Green_g       0.10   -0.0041   -0.0004
Dry_Dead_g        0.10   -0.0403   -0.0040
Dry_Clover_g      0.10   -0.0428   -0.0043
GDM_g             0.20   -0.0639   -0.0128
Dry_Total_g       0.50   -0.0378   -0.0189
                                 ---------
WEIGHTED                           -0.0404

pooled R2 in grams, same predictions: 0.1927   <-- the old metric


In [7]:
# Feature views. DIM is read from the manifest, never hardcoded.

DIM = manifest['feature_dim']
VIEWS = {
    'cls_id'        : F[:, 0, :DIM],
    'patch_id'      : F[:, 0, DIM:],
    'both_id'       : F[:, 0, :],
    'both_flipavg'  : F.mean(axis=1),
    'both_allviews' : F.reshape(len(F), -1),
}
for k, v in VIEWS.items():
    print(f'{k:16} {v.shape}')

cls_id           (357, 1280)
patch_id         (357, 1280)
both_id          (357, 2560)
both_flipavg     (357, 2560)
both_allviews    (357, 10240)


In [8]:
# Head + trainer. Targets are log1p then standardized; the loss is per-target
# squared error weighted by the COMPETITION weights [.1 .1 .1 .2 .5].
#
# Why this loss: R2 = 1 - SSE/SS_tot, so minimising weighted squared error on
# log1p targets directly maximises the weighted log-space R2 that is scored.
# In the old long format every target got 20% of the gradient, which starved
# Dry_Total -- worth 50% of the score -- by a factor of 2.5.

W = torch.tensor([WEIGHTS[t] for t in TARGETS], device=device)

class Head(nn.Module):
    def __init__(self, d_in, hidden=512, p=0.3, n_out=5):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(d_in),
            nn.Linear(d_in, hidden), nn.GELU(), nn.Dropout(p),
            nn.Linear(hidden, n_out),
        )
    def forward(self, x):
        return self.net(x)


def run_variant(X, y_log, folds, epochs=500, hidden=512, p=0.3, lr=1e-3,
                wd=1e-2, seeds=(0, 1, 2), verbose=False):
    """Returns (oof_grams, mean_score). oof is averaged over seeds."""
    Xg = torch.tensor(np.asarray(X, dtype=np.float32), device=device)
    Yg = torch.tensor(y_log, dtype=torch.float32, device=device)

    oof = torch.zeros_like(Yg)
    for seed in seeds:
        for k in range(folds.max() + 1):
            tr = torch.tensor(folds != k, device=device)
            va = torch.tensor(folds == k, device=device)

            mu, sd = Yg[tr].mean(0), Yg[tr].std(0).clamp_min(1e-6)
            ytr = (Yg[tr] - mu) / sd

            torch.manual_seed(seed)
            m   = Head(Xg.shape[1], hidden, p).to(device)
            opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=wd)
            sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

            xtr = Xg[tr]
            for _ in range(epochs):
                m.train(); opt.zero_grad()
                loss = (((m(xtr) - ytr) ** 2).mean(0) * W).sum()
                loss.backward(); opt.step(); sch.step()

            m.eval()
            with torch.no_grad():
                oof[va] += m(Xg[va]) * sd + mu

    oof /= len(seeds)
    oof_g = np.expm1(oof.cpu().numpy()).clip(0, None)
    score, per_t = weighted_log_r2(wide[TARGETS], oof_g, verbose=verbose)
    return oof_g, score, per_t


y_log = np.log1p(wide[TARGETS].values)
print('trainer ready')

trainer ready


In [16]:
# Head training runs on CPU: it is a 357-row matrix multiply, ~90s per variant.
# (A timm upgrade also broke this session's CUDA build, but the GPU was never
#  needed for this phase — only for the feature extraction in notebook 1.)
device = torch.device('cpu')
W = torch.tensor([WEIGHTS[t] for t in TARGETS], device=device)
print('device is now:', device)

device is now: cpu


In [11]:
# EXPERIMENT 1 -- which feature view? Five variants, seconds each.

print(f'{"view":16} {"weighted logR2":>15} {"vs floor":>10} {"pooled(old)":>12}')
print('-' * 56)
results = {}
for name, X in VIEWS.items():
    t0 = time.time()
    oof, score, _ = run_variant(X, y_log, folds)
    results[name] = (score, oof)
    print(f'{name:16} {score:15.4f} {score-base_score:+10.4f} '
          f'{pooled_r2_grams(wide[TARGETS], oof):12.4f}   [{time.time()-t0:.1f}s]')

best_view = max(results, key=lambda k: results[k][0])
print(f'\nbest view: {best_view}  ({results[best_view][0]:.4f})')

view              weighted logR2   vs floor  pooled(old)
--------------------------------------------------------
cls_id                    0.6086    +0.6490       0.7213   [93.3s]
patch_id                  0.6755    +0.7159       0.7315   [89.0s]
both_id                   0.6547    +0.6951       0.7431   [152.4s]
both_flipavg              0.6737    +0.7142       0.7468   [152.4s]
both_allviews             0.6605    +0.7009       0.7321   [571.0s]

best view: patch_id  (0.6755)


In [12]:
# EXPERIMENT 2 -- head capacity and regularisation, on the winning view.

X = VIEWS[best_view]
print(f'{"hidden":>7} {"dropout":>8} {"weighted logR2":>15}')
print('-' * 32)
grid = {}
for hidden, p in itertools.product([256, 512, 1024], [0.1, 0.3, 0.5]):
    _, score, _ = run_variant(X, y_log, folds, hidden=hidden, p=p)
    grid[(hidden, p)] = score
    print(f'{hidden:7d} {p:8.1f} {score:15.4f}')

best_cfg = max(grid, key=grid.get)
print(f'\nbest: hidden={best_cfg[0]} dropout={best_cfg[1]}  ({grid[best_cfg]:.4f})')

 hidden  dropout  weighted logR2
--------------------------------
    256      0.1          0.6732
    256      0.3          0.6778
    256      0.5          0.6832
    512      0.1          0.6638
    512      0.3          0.6755
    512      0.5          0.6839
   1024      0.1          0.6633
   1024      0.3          0.6730
   1024      0.5          0.6822

best: hidden=512 dropout=0.5  (0.6839)


In [13]:
# EXPERIMENT 3 -- does LUPI help?
# NDVI and Height exist in train.csv but NOT in test.csv. Predict them from
# the frozen features, then feed the PREDICTIONS in alongside the features.
# This is the 4th-place team's auxiliary trick, tested for free.

aux_cols = ['Pre_GSHH_NDVI', 'Height_Ave_cm']
A = wide[aux_cols].values.astype(np.float32)

Xg = torch.tensor(np.asarray(X, dtype=np.float32), device=device)
Ag = torch.tensor(A, device=device)
aux_oof = torch.zeros_like(Ag)

for k in range(folds.max() + 1):
    tr = torch.tensor(folds != k, device=device)
    va = torch.tensor(folds == k, device=device)
    mu, sd = Ag[tr].mean(0), Ag[tr].std(0).clamp_min(1e-6)
    torch.manual_seed(0)
    m   = Head(Xg.shape[1], 512, 0.3, n_out=len(aux_cols)).to(device)
    opt = torch.optim.AdamW(m.parameters(), lr=1e-3, weight_decay=1e-2)
    for _ in range(500):
        m.train(); opt.zero_grad()
        ((m(Xg[tr]) - (Ag[tr] - mu) / sd) ** 2).mean().backward(); opt.step()
    m.eval()
    with torch.no_grad():
        aux_oof[va] = m(Xg[va]) * sd + mu

aux_np = aux_oof.cpu().numpy()
print('how well can the image predict the hidden metadata?')
from sklearn.metrics import r2_score
for i, c in enumerate(aux_cols):
    print(f'  {c:16} R2 {r2_score(A[:, i], aux_np[:, i]):7.4f}')

X_lupi = np.hstack([np.asarray(X, dtype=np.float32),
                    (aux_np - aux_np.mean(0)) / (aux_np.std(0) + 1e-6)])
_, s_lupi, _ = run_variant(X_lupi, y_log, folds,
                           hidden=best_cfg[0], p=best_cfg[1])
print(f'\nwithout LUPI  {grid[best_cfg]:.4f}')
print(f'with    LUPI  {s_lupi:.4f}   delta {s_lupi-grid[best_cfg]:+.4f}')
print('\n(your measurement floor is about +/-0.025 -- treat anything smaller as noise)')

how well can the image predict the hidden metadata?
  Pre_GSHH_NDVI    R2  0.8841
  Height_Ave_cm    R2  0.8303

without LUPI  0.6839
with    LUPI  0.6872   delta +0.0033

(your measurement floor is about +/-0.025 -- treat anything smaller as noise)


In [14]:
# FINAL: best configuration, full per-target report, saved for notebook 3.

X_final = X_lupi if s_lupi > grid[best_cfg] else X
oof, score, per_t = run_variant(X_final, y_log, folds, hidden=best_cfg[0],
                                p=best_cfg[1], seeds=(0,1,2,3,4), verbose=True)

print(f'\nfloor              {base_score:+.4f}')
print(f'frozen-feature head {score:+.4f}')
print(f'pooled (old metric) {pooled_r2_grams(wide[TARGETS], oof):+.4f}')

np.save('/kaggle/working/oof_frozen.npy', oof)
json.dump({'view': best_view, 'hidden': best_cfg[0], 'dropout': best_cfg[1],
           'lupi': bool(s_lupi > grid[best_cfg]), 'score': float(score),
           'floor': float(base_score), 'per_target': {k: float(v) for k, v in per_t.items()}},
          open('/kaggle/working/frozen_config.json', 'w'), indent=2)
print('\nsaved oof_frozen.npy + frozen_config.json')
print('This is the number to beat with fine-tuning in notebook 3.')

target          weight   R2(log)   contrib
Dry_Green_g       0.10    0.7125    0.0712
Dry_Dead_g        0.10    0.4591    0.0459
Dry_Clover_g      0.10    0.7615    0.0761
GDM_g             0.20    0.7026    0.1405
Dry_Total_g       0.50    0.7045    0.3522
                                 ---------
WEIGHTED                            0.6861

floor              -0.0404
frozen-feature head +0.6861
pooled (old metric) +0.7304

saved oof_frozen.npy + frozen_config.json
This is the number to beat with fine-tuning in notebook 3.
